In [1]:
resolution = 3
# flex_path = "../results/20250414-NEP-flex/8Gt_Bal_v3/networks/base_s_27__720H_" #base_s_27_lvopt__6H_2045.nc
# gas_path = "../results/NEP_gas_3h/8Gt_Bal_v3/postnetworks/base_s_27_lvopt__6H_" #base_s_27_lvopt__6H_2045.nc

# gas_path = "./result/cc-3-gas/base_s_27__3H_"
gas_path = "./result/cc-4-gas/base_s_27__3H_"
# flex_path = "./result/cc-4-flex/base_s_27__3H_"
flex_path = "../results/20250414-NEP-flex/8Gt_Bal_v3/networks/base_s_27__3H_"



scenario1_name = 'Gas'
scenario2_name = 'Flex'

In [2]:
import pypsa
import pandas as pd
from pea import Pea
from pea import preset as pp
import numpy as np

import plotly.express as px
import plotly.io as pio
import plotly.graph_objects as go
import plotly

pio.templates.default = 'seaborn' #'plotly_white' #'seaborn' #https://plotly.com/python/templates/
pd.options.plotting.backend = "plotly"


%load_ext autoreload
%autoreload 2

ppv = vars(pp)

In [3]:
nf45 = pypsa.Network(f"{flex_path}2045.nc")

       'DE0 1 CCGT-2010', 'DE0 1 CCGT-2015', 'DE0 1 OCGT-2015',
       'DE0 2 CCGT-1980', 'DE0 2 CCGT-2020', 'DE0 4 CCGT-2015',
       'DE0 4 CCGT-2020', 'DE0 6 oil-1975', 'DE0 7 CCGT-1965',
       'DE0 7 CCGT-2015'],
      dtype='object', name='Link') for attribute p0 of Link are not in main components dataframe links
       'DE0 1 CCGT-2010', 'DE0 1 CCGT-2015', 'DE0 1 OCGT-2015',
       'DE0 2 CCGT-1980', 'DE0 2 CCGT-2020', 'DE0 4 CCGT-2015',
       'DE0 4 CCGT-2020', 'DE0 6 oil-1975', 'DE0 7 CCGT-1965',
       'DE0 7 CCGT-2015'],
      dtype='object', name='Link') for attribute mu_lower of Link are not in main components dataframe links
       'DE0 1 CCGT-2010', 'DE0 1 CCGT-2015', 'DE0 1 OCGT-2015',
       'DE0 2 CCGT-1980', 'DE0 2 CCGT-2020', 'DE0 4 CCGT-2015',
       'DE0 4 CCGT-2020', 'DE0 6 oil-1975', 'DE0 7 CCGT-1965',
       'DE0 7 CCGT-2015'],
      dtype='object', name='Link') for attribute mu_upper of Link are not in main components dataframe links
INFO:pypsa.io:Imported ne

In [7]:
buses = nf45.buses.index[(nf45.buses.index.str[:2] == "DE")]
balance = nf45.statistics.energy_balance(
                aggregate_time=False,
                nice_names=False,
                groupby=pypsa.statistics.groupers["bus", "carrier", "bus_carrier"],
            ).loc[:, buses, :, :].droplevel("bus")

nodal_balance = balance
carriers=["AC", "low voltage", "DSM"]
# carriers = ["H2"]
mask = nodal_balance.index.get_level_values("bus_carrier").isin(carriers)
mask
nb = balance[mask].groupby("carrier").sum().div(1e3).T['2019-01-01 00:00:00': '2019-01-31 00:00:00']
px.bar(nb)

In [5]:
nb

carrier,Fischer-Tropsch,H2 Electrolysis,H2 Fuel Cell,H2 OCGT,H2 Store,H2 for industry,H2 pipeline,H2 pipeline (Kernnetz),H2 pipeline retrofitted,SMR,SMR CC,Sabatier,land transport fuel cell,load,methanolisation
snapshot,,,,,,,,,,,,,,,
2019-01-01 00:00:00,-0.000133,3.040737e+01,-1.120000e-06,-0.000003,-12.500650,-7.929224,-7.605065,3.362357,-3.886373,0.0,0.0,-0.000012,-0.138172,1.600000e-07,-1.710098
2019-01-01 03:00:00,-0.000136,3.040738e+01,-1.050000e-06,-0.000003,-8.070059,-7.929224,-7.605270,3.143258,-4.479367,0.0,0.0,-0.000012,-0.687940,1.600000e-07,-4.778622
2019-01-01 06:00:00,-0.000144,3.040739e+01,-9.800000e-07,-0.000003,-5.858096,-7.929224,-7.742709,1.755704,-4.396340,0.0,0.0,-0.000012,-1.457872,1.600000e-07,-4.778695
2019-01-01 09:00:00,-0.000146,3.040740e+01,-9.200000e-07,-0.000003,-1.738928,-7.929224,-7.705452,-2.455412,-4.424830,0.0,0.0,-0.000012,-1.198376,1.600000e-07,-4.955017
2019-01-01 12:00:00,-0.000146,3.040740e+01,-9.300000e-07,-0.000003,-1.150578,-7.929224,-7.744139,-2.819025,-4.402133,0.0,0.0,-0.000012,-1.407123,1.600000e-07,-4.955017
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2019-01-30 12:00:00,-0.000050,3.290000e-06,-1.840000e-06,-0.000004,1.320897,-7.929224,-1.231447,5.221606,5.822102,0.0,0.0,-0.000011,-1.493861,1.600000e-07,-1.710010
2019-01-30 15:00:00,-0.000046,3.300000e-06,-2.250000e-06,-0.000004,20.384467,-7.929224,-3.778452,-2.008564,-3.292067,0.0,0.0,-0.000011,-1.666090,1.600000e-07,-1.710010
2019-01-30 18:00:00,-0.000045,3.090000e-06,-6.370000e-06,-2.591720,13.919390,-7.929224,-3.633223,4.099629,-1.364766,0.0,0.0,-0.000011,-0.790020,1.600000e-07,-1.710007


In [6]:
n = nf45
m = n.optimize.create_model()

Index(['DE0 0 urban central gas CHP-2010', 'DE0 1 urban central gas CHP-2010',
       'DE0 2 urban central gas CHP-2010', 'DE0 3 urban central gas CHP-2010',
       'DE0 4 urban central gas CHP-2010', 'DE0 5 urban central gas CHP-2010',
       'DE0 6 urban central gas CHP-2010', 'DE0 7 urban central gas CHP-2010',
       'DE0 2 urban central oil CHP-2010', 'DE0 4 urban central oil CHP-2010',
       ...
       'DE0 6 H2 OCGT-2040', 'DE0 7 H2 OCGT-2040', 'DE0 0 H2 OCGT-2045',
       'DE0 1 H2 OCGT-2045', 'DE0 2 H2 OCGT-2045', 'DE0 3 H2 OCGT-2045',
       'DE0 4 H2 OCGT-2045', 'DE0 5 H2 OCGT-2045', 'DE0 6 H2 OCGT-2045',
       'DE0 7 H2 OCGT-2045'],
      dtype='object', name='Link', length=139)
